In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from scipy.stats import linregress, ks_2samp

import json
import sys
from pathlib import Path

In [ ]:
sys.path.append(str(Path.cwd().parent))

from src.config import target

In [ ]:
df = pd.read_csv("../data/combined_gee_pm25.csv")

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df = df.dropna(subset=["date"]).copy()

print("Dataset shape:", df.shape)
print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

In [ ]:
df = df.sort_values(
    ["sensor_name", "date"]
).reset_index(drop=True)

print("\nTimestamp quality")

print("Missing dates:", df["date"].isna().sum())

print(
    "Duplicate sensor-date pairs:",
    df.duplicated(
        subset=["sensor_name", "date"]
    ).sum()
)

In [ ]:
daily_series = (
    df.groupby([
        "sensor_name",
        pd.Grouper(key="date", freq="D")
    ])[target]
    .median()
    .reset_index()
)

daily_series = daily_series.dropna(
    subset=[target]
)

print(daily_series.head())
print("Number of daily observations:", len(daily_series))

In [ ]:
top_sensor = (
    daily_series["sensor_name"]
    .value_counts()
    .index[0]
)

sensor_data = (
    daily_series[
        daily_series["sensor_name"] == top_sensor
    ]
    .sort_values("date")
)

series = sensor_data[target].dropna()

print("Selected sensor:", top_sensor)
print("Number of observations:", len(series))

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 8)
)

plot_acf(
    series,
    lags=min(30, len(series) - 1),
    ax=axes[0]
)

axes[0].set_title(
    f"ACF of PM2.5 - {top_sensor}"
)

plot_pacf(
    series,
    lags=min(20, len(series) // 2 - 1),
    ax=axes[1],
    method="ywm"
)

axes[1].set_title(
    f"PACF of PM2.5 - {top_sensor}"
)

plt.tight_layout()
plt.show()

In [ ]:
adf_result = adfuller(series)

print("ADF Statistic:", round(adf_result[0], 4))
print("p-value:", round(adf_result[1], 4))

if adf_result[1] < 0.05:
    print(
        "The series is stationary "
        "(p-value < 0.05)."
    )
else:
    print(
        "The series may not be stationary "
        "(p-value >= 0.05)."
    )

In [ ]:
adf_result = adfuller(series)

print("ADF Statistic:", round(adf_result[0], 4))
print("p-value:", round(adf_result[1], 4))

if adf_result[1] < 0.05:
    print(
        "The series is stationary "
        "(p-value < 0.05)."
    )
else:
    print(
        "The series may not be stationary "
        "(p-value >= 0.05)."
    )

In [ ]:
ljung_box = acorr_ljungbox(
    series,
    lags=[10],
    return_df=True
)

print(ljung_box)

if ljung_box["lb_pvalue"].iloc[0] < 0.05:
    print(
        "There is significant autocorrelation."
    )
else:
    print(
        "No significant autocorrelation was detected."
    )

In [ ]:
sensor_data = sensor_data.reset_index(
    drop=True
)

sensor_data["time_index"] = np.arange(
    len(sensor_data)
)

trend_result = linregress(
    sensor_data["time_index"],
    sensor_data[target]
)

print(
    "Trend slope:",
    round(trend_result.slope, 4)
)

print(
    "Trend p-value:",
    round(trend_result.pvalue, 4)
)

plt.figure(figsize=(12, 5))

plt.plot(
    sensor_data["date"],
    sensor_data[target],
    alpha=0.6,
    label="Daily PM2.5"
)

trend_line = (
    trend_result.intercept +
    trend_result.slope *
    sensor_data["time_index"]
)

plt.plot(
    sensor_data["date"],
    trend_line,
    label="Linear trend"
)

plt.title(
    f"PM2.5 Trend - {top_sensor}"
)

plt.xlabel("Date")
plt.ylabel("PM2.5")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
df["month"] = df["date"].dt.month

monthly_stats = (
    df.groupby("month")[target]
    .agg(["count", "mean", "median", "std"])
)

print(monthly_stats.round(2))

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df,
    x="month",
    y=target
)

plt.title("Monthly PM2.5 Distribution")
plt.xlabel("Month")
plt.ylabel("PM2.5")

plt.tight_layout()
plt.show()

In [ ]:
monthly_missing = (
    df.assign(month=df["date"].dt.month)
    .groupby("month")[target]
    .apply(lambda x: x.isna().mean() * 100)
)

print(
    "\nMissing PM2.5 percentage by month:"
)

print(monthly_missing.round(2))

In [ ]:
lag_variables = [
    "AOD",
    "NO2",
    "CO",
    "O3"
]

lag_results = []

for sensor in df["sensor_name"].dropna().unique():

    sensor_df = (
        df[df["sensor_name"] == sensor]
        .sort_values("date")
        .copy()
    )

    for variable in lag_variables:

        if variable not in sensor_df.columns:
            continue

        sensor_df[f"{variable}_lag_1"] = (
            sensor_df[variable].shift(1)
        )

        correlation_data = sensor_df[
            [target, f"{variable}_lag_1"]
        ].dropna()

        if len(correlation_data) > 2:

            correlation = correlation_data[
                target
            ].corr(
                correlation_data[
                    f"{variable}_lag_1"
                ]
            )

            lag_results.append({
                "sensor_name": sensor,
                "variable": variable,
                "lag": 1,
                "correlation": correlation,
                "observations": len(
                    correlation_data
                )
            })

lead_lag_results = pd.DataFrame(
    lag_results
)

print(lead_lag_results.head())